# 1. Setup

DINOv2 partial fine-tuning with disentanglement and strict cross-crop SupCon. Each trial unfreezes the last 2, 4, 6, or 8 transformer blocks in **each** of the two Baseline 1 visual towers. The remaining backbone weights stay frozen. Backbone learning rate is 1e-5.

In [ ]:
import gc, json, math, sys
from pathlib import Path
from datetime import datetime
import timm, torch
import pandas as pd
from torch import nn
from timm.layers import set_reentrant_ckpt
from torch.nn import functional as F
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'common/dataset.py').is_file())
sys.path.insert(0, str(ROOT))
from common import config as cfg
from common.dataset import get_loaders, get_pairs, get_crop_and_disease_names, load_split, get_final_test_loaders
from common.evaluate import evaluate, val_accuracy, concept_accuracy
MODEL = 'vit_base_patch14_dinov2.lvd142m'
MODEL_NAME, FAMILY = 'approach3a_dino_disentangle_supcon_v2', 'A'
DEBUG = True
HP = dict(backbone=MODEL, batch_size=8 if DEBUG else 16, epochs=1 if DEBUG else 20,
          unfreeze_blocks=[2, 4, 6, 8], backbone_lr=1e-5,
          crop_lr=1e-4, disease_lr=3e-4, weight_decay=1e-4, label_smoothing=0.1,
          seed=cfg.SEED, norm='imagenet', hidden_dim=256, feature_dim=128,
          supcon_temperature=0.1, text_batch_size=16, dino_revision='main', text_revision='main',
          score_weights=[0.5,0.25,0.25],
          pair_templates=['a photo of a {crop} leaf with {disease}',
                          'a field photo of {crop} showing {disease} symptoms',
                          'a close-up leaf of {crop} affected by {disease}'],
          healthy_templates=['a photo of a healthy {crop} leaf','a field photo of a healthy {crop} plant'])
EPOCHS = HP['epochs']
cfg.NUM_WORKERS = 0  # Exact ToTnki runtime setting; common files are unchanged.
cfg.set_seed(cfg.SEED)
RUN = MODEL_NAME + ('_debug' if DEBUG else '') + '_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f')
OUT = cfg.CHECKPOINT_DIR / RUN
environment = dict(python=sys.version, torch=str(torch.__version__), timm=str(timm.__version__), device=str(cfg.DEVICE))

# 2. Data
Direct common loaders: formal7223 train/777 validation,323 development pairs. DEBUG uses common subsets. No extra holdout, sampler or augmentation. No official test CSV is read here.


In [ ]:
# Same common data path as the old CLIP notebook; DINO uses imagenet normalization.
frames = [load_split(cfg.TRAIN_CSV)]
assert len(frames[0]) == 8000
pairs = get_pairs()
assert pairs.shape == (cfg.NUM_SEEN_PAIRS, 2)
concept_names = get_crop_and_disease_names()
def loaders():
    cfg.set_seed(cfg.SEED)
    return get_loaders(HP['batch_size'], norm=HP['norm'], debug=DEBUG)
train_loader, val_loader, seen_loader, unseen_loader = loaders()
for loader in (train_loader, val_loader): loader.pin_memory = False
assert not set(train_loader.dataset.frame.image) & set(val_loader.dataset.frame.image)
sample_counts = dict(train=len(train_loader.dataset), val=len(val_loader.dataset))
print('Device:', cfg.DEVICE, 'Data:', cfg.DATA_DIR, 'Samples:', sample_counts)


# 3. Model

Two DINOv2 visual towers with disentanglement and strict cross-crop SupCon. Activation checkpointing reduces training memory on the selected blocks.

In [ ]:
def snapshot(repository,revision,allow_download,patterns):
    from huggingface_hub import snapshot_download
    path=Path(snapshot_download(repository,revision=revision,allow_patterns=patterns,local_files_only=not allow_download))
    return path

class Dino(nn.Module):
    """Exact two-independent-tower initialization from ToTnki's released runner."""
    def __init__(self,config,device,allow_download=False,revision=None):
        super().__init__()
        import timm
        path=snapshot('timm/vit_base_patch14_dinov2.lvd142m',revision or config['dino_revision'],allow_download,
                      ['config.json','model.safetensors','pytorch_model.bin'])
        weights=next(path/name for name in ['model.safetensors','pytorch_model.bin'] if (path/name).is_file())
        def pinned_source_model(classes):
            # Same source factory, with only an explicit weight-file binding added.
            model=timm.create_model(MODEL,pretrained=True,num_classes=classes,img_size=cfg.IMAGE_SIZE,
                                    pretrained_cfg_overlay={'file':str(weights)})
            model.requires_grad_(False);model.head.requires_grad_(True)
            return model
        self.crop=pinned_source_model(cfg.NUM_CROPS)
        self.disease=pinned_source_model(cfg.NUM_DISEASES)
        self.augmentation_rng_state=torch.get_rng_state().clone()
        self.width=self.crop.num_features
        self.metadata=dict(repository='timm/vit_base_patch14_dinov2.lvd142m',revision=path.name,
            architecture=MODEL,width=self.width,independent_visual_towers=2,
            initialization='ToTnki source factory settings, crop then disease, explicitly pinned public weight file',
            preprocessing='common imagenet; img_size224; timm native position resampling and pre_logits')
        self.to(device)
    def policy(self, depth):
        self.requires_grad_(False)
        for tower in (self.crop, self.disease):
            tower.blocks[-depth:].requires_grad_(True)
            tower.set_grad_checkpointing(True)

    def features(self,pixels):
        return (self.crop.forward_head(self.crop.forward_features(pixels),pre_logits=True).float(),
                self.disease.forward_head(self.disease.forward_features(pixels),pre_logits=True).float())
    def forward(self,pixels):return self.features(pixels)

def projection(width,hidden,out):return nn.Sequential(nn.Linear(width,hidden),nn.ReLU(),nn.Linear(hidden,out))

def positive_mask(crops, diseases, cross_crop=True):
    result = diseases[:, None] == diseases[None, :]
    result &= ~torch.eye(len(crops), dtype=torch.bool, device=crops.device)
    if cross_crop: result &= crops[:, None] != crops[None, :]
    return result

def supcon(features, crops, diseases, temperature=.1, cross_crop=True):
    features = F.normalize(features.float(), dim=1)
    positives = positive_mask(crops, diseases, cross_crop)
    valid = positives.any(1)
    if not valid.any(): return features.sum()*0, 0.
    diagonal = torch.eye(len(features), dtype=torch.bool, device=features.device)
    logits = (features @ features.T / temperature).masked_fill(diagonal, -torch.inf)
    log_prob = logits - torch.logsumexp(logits, dim=1, keepdim=True)
    # masked_fill avoids 0 * -inf on the diagonal.
    means = log_prob.masked_fill(~positives, 0).sum(1) / positives.sum(1).clamp_min(1)
    return -means[valid].mean(), valid.float().mean().item()

def decorrelation(crop, disease):
    if len(crop) < 2: return (crop.sum()+disease.sum())*0
    crop, disease = crop.float(), disease.float()
    def standardize(x): return (x-x.mean(0)) / x.std(0, unbiased=True).clamp_min(1e-6)
    covariance = standardize(crop).T @ standardize(disease) / (len(crop)-1)
    return covariance.square().mean()

def pair_scores(crop, disease, pairs):
    pairs = pairs.to(crop.device)
    return crop.log_softmax(1)[:, pairs[:, 0]] + disease.log_softmax(1)[:, pairs[:, 1]]


RECIPE = dict(sc=.1, dis=.01)


class Model(nn.Module):
    def __init__(self, backbone, banks):
        super().__init__()
        self.backbone = backbone
        width, hidden = backbone.width, HP['hidden_dim']
        dim = HP['feature_dim']
        self.crop_projection = projection(width, hidden, dim)
        self.disease_projection = projection(width, hidden, dim)

        self.crop_head = nn.Linear(dim, cfg.NUM_CROPS)
        self.disease_head = nn.Linear(dim, cfg.NUM_DISEASES)

    def forward(self, pixels):
        crop_features, disease_features = self.backbone(pixels)
        crop = F.normalize(self.crop_projection(crop_features), dim=1)
        disease = F.normalize(self.disease_projection(disease_features), dim=1)
        return self.crop_head(crop), self.disease_head(disease), crop, disease


def loss(output, labels):
    crop, disease, crop_features, disease_features = output
    crops, diseases = pairs.to(labels.device)[labels].unbind(1)
    smoothing = HP['label_smoothing']
    classification = (F.cross_entropy(crop, crops, label_smoothing=smoothing)
                      + F.cross_entropy(disease, diseases, label_smoothing=smoothing))
    contrastive, anchors = supcon(disease_features, crops, diseases, HP['supcon_temperature'])
    return classification + RECIPE['sc'] * contrastive + RECIPE['dis'] * decorrelation(crop_features, disease_features), anchors

def make_model(pairs, names, depth, metadata=None):
    cfg.set_seed(cfg.SEED)
    metadata = metadata or {}
    backbone = Dino(HP, cfg.DEVICE, allow_download=True, revision=metadata.get('dino', {}).get('revision'))
    backbone.policy(depth)
    text, banks = None, None

    torch.manual_seed(cfg.SEED)
    model = Model(backbone, banks).to(cfg.DEVICE)
    torch.set_rng_state(backbone.augmentation_rng_state)
    return model, text, dict(dino=backbone.metadata, text=text.metadata if text else None)


def make_predict(model, candidate_pairs):
    @torch.no_grad()
    def predict(images):
        model.eval()
        return pair_scores(*model(images.to(cfg.DEVICE))[:2], candidate_pairs)
    return predict

# 4. Training

Restart from the same pretrained revisions and seed at each depth. Each trial gets the same epoch budget and shared train/validation split. Keep the highest validation Top-1, preferring fewer unfrozen blocks on ties and the earlier epoch within that depth. Save only the overall winning weights. Official tests remain unopened.

In [ ]:
def train_trial(depth, revisions, best, checkpoint_path):
    train_loader, val_loader, _, _ = loaders()
    train_loader.pin_memory = val_loader.pin_memory = False
    model, text, metadata = make_model(pairs, concept_names, depth, revisions)
    del text  # Training uses the already computed frozen text embeddings.
    groups = {'backbone': [], 'crop': [], 'disease': []}
    for name, parameter in model.named_parameters():
        if parameter.requires_grad:
            key = 'backbone' if name.startswith('backbone.') else 'disease' if 'disease' in name else 'crop'
            groups[key].append(parameter)
    optimizer = torch.optim.AdamW(
        [dict(params=groups[key], lr=HP[key + '_lr']) for key in groups],
        weight_decay=HP['weight_decay'])
    schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    predict = make_predict(model, pairs)
    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        model.backbone.eval()
        for tower in (model.backbone.crop, model.backbone.disease):
            tower.blocks[-depth:].train()
        total_loss = anchor_count = samples = 0
        for images, _, _, labels in train_loader:
            optimizer.zero_grad(set_to_none=True)
            values = model(images.to(cfg.DEVICE))
            value, anchor_fraction = loss(values, labels.to(cfg.DEVICE))
            if not torch.isfinite(value):
                raise RuntimeError('Nonfinite training loss.')
            value.backward()
            optimizer.step()
            samples += len(labels)
            total_loss += value.item() * len(labels)
            anchor_count += anchor_fraction * len(labels)
        schedule.step()
        validation = val_accuracy(predict, val_loader)
        record = dict(depth=depth, epoch=epoch, val_top1=validation, loss=total_loss / samples)
        if FAMILY == 'A':
            record['valid_crosscrop_anchor_fraction'] = anchor_count / samples
        history.append(record)
        print(record)
        if best is None or validation > best['val_top1']:
            best = record
            torch.save(dict(identity, depth=depth, best_val=validation, best_epoch=epoch, pairs=pairs,
                            metadata=metadata, state_dict=model.state_dict()), checkpoint_path)
    return best, history, metadata

OUT.mkdir(parents=True, exist_ok=False)
set_reentrant_ckpt(False)  # The frozen prefix produces tensors without gradients.
identity = dict(model_name=MODEL_NAME, family=FAMILY, debug=DEBUG, hp=HP, recipe=RECIPE)
checkpoint_path = OUT / (MODEL_NAME + '.pt')
best, revisions, trials = None, None, []
for depth in HP['unfreeze_blocks']:
    best, history, revisions = train_trial(depth, revisions, best, checkpoint_path)
    trials.append(max(history, key=lambda record: record['val_top1']))
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
selection = dict(identity, checkpoint=checkpoint_path.name, depth=best['depth'],
                 best_val=best['val_top1'], best_epoch=best['epoch'], trials=trials,
                 train_images=train_loader.dataset.frame.image.tolist(),
                 validation_images=val_loader.dataset.frame.image.tolist(),
                 development_pairs=pairs.tolist())
(OUT / 'selection.json').write_text(json.dumps(selection, indent=2), encoding='utf8')
display(pd.DataFrame(trials))
print(f"Selected {best['depth']} blocks per tower at epoch {best['epoch']} ({best['val_top1']:.2f}% val Top-1).")

# 5. Evaluation

After both approaches' choices are frozen, enable final evaluation for the validation winner. Official candidate pairs and test loaders are opened only after it is loaded. The shared evaluator reports Top-1, Top-5, harmonic mean, seen confusion and robustness. Unseen crop and disease accuracy use the same clean unseen-combination predictions.

In [ ]:
RUN_FINAL_EVALUATION = False  # Enable only after all model choices are frozen.
reports = []
if RUN_FINAL_EVALUATION:
    if DEBUG:
        raise ValueError('Official evaluation requires a full non-DEBUG run.')
    run_folder = OUT
    selection = json.loads((run_folder / 'selection.json').read_text(encoding='utf8'))
    checkpoint = torch.load(run_folder / selection['checkpoint'], map_location='cpu', weights_only=True)
    expected = dict(model_name=MODEL_NAME, family=FAMILY, debug=False, hp=HP, recipe=RECIPE)
    if any(record.get(key) != value for record in (selection, checkpoint)
           for key, value in expected.items()):
        raise ValueError('The selected run or checkpoint does not match the frozen settings.')
    if selection['development_pairs'] != pairs.tolist() or not torch.equal(checkpoint['pairs'], pairs):
        raise ValueError('Development candidates changed.')
    if (selection['train_images'] != train_loader.dataset.frame.image.tolist()
            or selection['validation_images'] != val_loader.dataset.frame.image.tolist()):
        raise ValueError('Common train/validation membership changed.')
    if (checkpoint['depth'], checkpoint['best_epoch'], checkpoint['best_val']) != (
            selection['depth'], selection['best_epoch'], selection['best_val']):
        raise ValueError('Checkpoint does not match the validation winner.')
    model, text, metadata = make_model(pairs, concept_names, checkpoint['depth'], checkpoint['metadata'])
    if metadata != checkpoint['metadata']:
        raise ValueError('Public pretrained model identity changed.')
    model.load_state_dict(checkpoint['state_dict'])
    model.eval().requires_grad_(False)
    for tower in (model.backbone.crop, model.backbone.disease):
        tower.set_grad_checkpointing(False)
    # Only now open the official candidates and test loaders.
    final_pairs = get_pairs(final_evaluation=True)
    seen_loader, unseen_loader = get_final_test_loaders(
        HP['batch_size'], norm=HP['norm'], final_evaluation=True)
    sample_counts.update(seen=len(seen_loader.dataset), unseen=len(unseen_loader.dataset))
    predict_final = make_predict(model, final_pairs)
    # common.evaluate predicts clean seen, then clean unseen, then robustness.
    seen_count = sample_counts['seen']
    clean_end = seen_count + sample_counts['unseen']
    unseen_batches, processed = [], 0
    def predict(images):
        global processed
        scores = predict_final(images)
        if seen_count <= processed < clean_end:
            unseen_batches.append(scores.float().cpu())
        processed += len(images)
        return scores
    RESULTS = cfg.RESULTS_DIR / run_folder.name
    RESULTS.mkdir(parents=True, exist_ok=False)
    previous_results = cfg.RESULTS_DIR
    try:
        cfg.RESULTS_DIR = RESULTS
        result = evaluate(predict, seen_loader, unseen_loader, MODEL_NAME, dict(HP, selected_depth=checkpoint['depth']), robust=True)
    finally:
        cfg.RESULTS_DIR = previous_results
    unseen_labels = torch.as_tensor(unseen_loader.dataset.frame.pair_row.to_numpy())
    crop_acc, disease_acc = concept_accuracy(torch.cat(unseen_batches), unseen_labels, final_evaluation=True)
    result.update(unseen_test_crop_acc=crop_acc, unseen_test_disease_acc=disease_acc,
                  debug=DEBUG, best_val=checkpoint['best_val'])
    reports.append(result)
    del predict_final, model, text, unseen_batches

# 6. Save

Keep the winning checkpoint and selection JSON with the validation comparison. Final evaluation also writes the official metrics and confusion files to this run's results folder.

In [ ]:
if RUN_FINAL_EVALUATION:
    for result in reports:
        (RESULTS/(result['model']+'.json')).write_text(json.dumps(result, indent=2), encoding='utf8')
    summary = dict(selection=selection, results=reports, hp=HP,
                   environment=environment, samples=sample_counts)
    (RESULTS/(MODEL_NAME+'_comparison.json')).write_text(json.dumps(summary, indent=2), encoding='utf8')
    table = pd.DataFrame(reports)
    table.to_csv(RESULTS/(MODEL_NAME+'_comparison.csv'), index=False)
    display(table[['model','seen_top1','unseen_top1','harmonic_mean',
                   'unseen_test_crop_acc','unseen_test_disease_acc']])
    print('Checkpoints:', run_folder, 'Results:', RESULTS)
else:
    print('Save the complete checkpoint run folder after training:', OUT)
